In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:22px;}
</style>
"""))

# 1. tensorflow v2.xx에서 v1 사용하기

In [3]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # tensorflow v2 비활성화하고 v1만 활성화
import numpy as np
import pandas as pd

Instructions for updating:
non-resource variables are not supported in the long term


## Tensorflow
- 데이터 흐름 그래프(tensor 흐름을 나타내는 설계도)를 사용하는 수치 계산 라이브러리
- 그래프는 node(데이터, 연산)와 edge로 구성
- sess = tf.Session()을 이용하여 실행환경
- sess.run()을 통해서 실행결과를 확인

In [8]:
# 1. tensor(상수node, 변수node, 연산node) 정의
node1 = tf.constant('Hello, Tensorflow')
# 2. 세션 생성(실행하는 환경 생성)
sess = tf.Session()
# 3. 실행
print(sess.run(node1))
print(sess.run(node1).decode())

b'Hello, Tensorflow'
Hello, Tensorflow


In [13]:
# 간단한 연산 tensor 그래프
# 1. 그래프 정의
node1 = tf.constant(10, dtype=tf.float16)
node2 = tf.constant(20, dtype=tf.float16)
node3 = tf.add(node1, node2)
# 2. 세션 생성
sess = tf.Session()
# 3. 세션 실행 및 결과
n1, n2, n3 = sess.run([node1, node2, node3])
print(n1, n2, n3)

10.0 20.0 30.0


In [14]:
# 타입 변경
node1 = tf.constant(np.array([10,20,30]), dtype=tf.int16 )
node2 = tf.cast(node1, dtype=tf.float32)
sess = tf.Session()
print(sess.run( [node1, node2] ))

[array([10, 20, 30], dtype=int16), array([10., 20., 30.], dtype=float32)]


In [21]:
# 평균값 계산 : tf.reduce_mean()
data = np.array([1., 2, 3, 4])
m = tf.reduce_mean(data)
sess = tf.Session()
sess.run(m)

2.5

In [28]:
# tf.random_normal([shape]) : 평균0, 표준편차는 1인 shape 난수 배열 tensor. 기본적으로 float32
w = tf.random.normal([1,3])
sess = tf.Session()
sess.run(w)

array([[-0.40631866,  0.0124842 ,  0.66786474]], dtype=float32)

In [31]:
# 변수노드
w = tf.Variable( tf.random.normal([1]) )
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # 난수가 발생될 변수 초기화
sess.run(w)

array([-0.7133757], dtype=float32)

# 2. tensorflow v1을 이용한 회귀분석 구현
## 2.1 독립(입력)변수 x가 1개, 종속(타겟)변수 y가 1개

In [36]:
# tensor 그래프 정의
# 데이터 셋 확보
x = np.array([1,2,3])
y = np.array([2,3,4])
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b])
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')
print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')

1번째 cost:15.3773193359375, w:[-0.4842809], b:[0.2391391]
301번째 cost:0.00027678292826749384, w:[1.0193225], b:[0.9561809]
601번째 cost:6.530730024678633e-05, w:[1.009386], b:[0.9786635]
901번째 cost:1.5410632840939797e-05, w:[1.0045594], b:[0.98963535]
1201번째 cost:3.6367002849146957e-06, w:[1.0022149], b:[0.9949651]
1501번째 cost:8.587500701651152e-07, w:[1.0010763], b:[0.99755347]
1801번째 cost:2.0313098048063694e-07, w:[1.0005233], b:[0.99881]
2101번째 cost:4.823188248792576e-08, w:[1.0002553], b:[0.9994203]
2401번째 cost:1.1453276549389102e-08, w:[1.0001243], b:[0.99971735]
2701번째 cost:2.7600453034182237e-09, w:[1.0000613], b:[0.9998614]
3001번째 cost:6.61752153074957e-10, w:[1.0000298], b:[0.99993205]
3301번째 cost:1.7466088297890536e-10, w:[1.0000155], b:[0.9999652]
3601번째 cost:5.409954081936341e-11, w:[1.0000088], b:[0.99998087]
3901번째 cost:5.409954081936341e-11, w:[1.0000088], b:[0.99998087]
4201번째 cost:5.409954081936341e-11, w:[1.0000088], b:[0.99998087]
4501번째 cost:5.409954081936341e-11, w:[1.

In [38]:
w_, b_ = sess.run([w[0], b[0]])
w_, b_

(1.0000088, 0.99998087)

In [39]:
def predict(x):
    return x*w_ + b_

In [40]:
predict(5)

6.000024974346161